# PCU-CROSS-LAYER-READOUT-001 — Frozen L7 association + minimal L23 readout

Engineering-only kill test. Reproduce the published L7/K64 hybrid association state, freeze it, allocate K16 Cells at L23 under that frozen state, train only L23 with the original answer-token CE, and compare against a matched-footprint L23-only control using the exact same 16 L23 Cell IDs.

Primary success requires cross-layer ranking >= 80%, direct greedy >= 80%, L23-only direct < 80%, and >= 30 percentage points direct synergy over the best single-layer control. Formal seeds are never executed.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-composability-kill-001'
REPO = Path('/kaggle/working/mini-cells')
OBJECTIVE = REPO / 'artifacts/research/pcu-objective-alignment-001/engineering/26090501-l7-k64-ranking'
HYBRID = REPO / 'artifacts/research/pcu-hybrid-objective-001/engineering/26090501-l7-k64-rank-plus-ce025'
READOUT = REPO / 'artifacts/research/pcu-readout-localization-001/engineering/26090501-l7-k64-hybrid-readout'
OUT = REPO / 'artifacts/research/pcu-cross-layer-readout-001/engineering/26090501-l7k64-plus-l23k16'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 1
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'hybrid_core_blob': run(['git', 'rev-parse', 'HEAD:src/minicells/pcu_kill_001/hybrid_objective.py'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'transformers': transformers.__version__,
}, indent=2))
assert run(['git', 'rev-parse', 'HEAD:src/minicells/pcu_kill_001/hybrid_objective.py'], capture=True) == '851c77cdd283def0698ebe721ea8bf216f5ed556'


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; token values were not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(row['seed']): row['state'] for row in payload['seeds']}
expected = {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert formal_states() == expected
assert run(['git', 'hash-object', SEED_REGISTRY], capture=True) == '71a3015a7d54e795538b3aa6750860f0b9168cb3'
print(json.dumps({'formal_seed_states': formal_states()}, indent=2))


In [ ]:
for root, names in ((OBJECTIVE, ['RESULT.json', 'DECISION.json', 'PAIRED_CE_K64.json']), (HYBRID, ['RESULT.json', 'DECISION.json']), (READOUT, ['RESULT.json', 'DECISION.json'])):
    missing = [name for name in names if not (root / name).is_file()]
    assert not missing, f'Missing prerequisite under {root}: {missing}'
hybrid_decision = json.loads((HYBRID / 'DECISION.json').read_text())
readout_decision = json.loads((READOUT / 'DECISION.json').read_text())
assert hybrid_decision['status'] == 'HYBRID_OBJECTIVE_PRESERVES_ASSOCIATION_GENERATION_UNRESOLVED'
assert abs(hybrid_decision['ranking_eval_accuracy'] - 0.8359375) < 1e-12
assert abs(hybrid_decision['direct_accuracy'] - 0.03125) < 1e-12
assert readout_decision['status'] == 'SINGLE_LAYER_GOLD_PREFIX_READOUT_INADEQUATE'
assert abs(readout_decision['later_token_top1_accuracy'] - 0.535031847133758) < 1e-12
for remote_path in [
    'artifacts/research/pcu-hybrid-objective-001/engineering/26090501-l7-k64-rank-plus-ce025/DECISION.json',
    'artifacts/research/pcu-readout-localization-001/engineering/26090501-l7-k64-hybrid-readout/DECISION.json',
]:
    assert subprocess.run(['git', 'show', f'origin/{BRANCH}:{remote_path}'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
print(json.dumps({'hybrid': hybrid_decision['status'], 'readout': readout_decision['status'], 'published': True}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO / 'src')
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001'], env=test_env)
run([sys.executable, '-m', 'compileall', '-q', 'src/minicells/pcu_kill_001', 'scripts/research'])
print('PCU cross-layer-readout test/compile gate: PASS')


In [ ]:
if OUT.exists():
    existing = sorted(p.name for p in OUT.glob('*.json'))
    assert not existing, f'Cross-layer output already exists; inspect before rerun: {existing}'
run([
    sys.executable, 'scripts/research/run_pcu_cross_layer_readout_001.py',
    '--seed', '26090501',
    '--device', 'cuda:0',
    '--objective-root', OBJECTIVE,
    '--hybrid-root', HYBRID,
    '--readout-root', READOUT,
    '--output', OUT,
])


In [ ]:
required = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json']
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing
decision = json.loads((OUT / 'DECISION.json').read_text())
result = json.loads((OUT / 'RESULT.json').read_text())
assert decision['valid_run'] is True
assert decision['formal_execution_not_started'] is True
assert decision['l7_reproduction_exact'] is True
assert decision['l23_selected_once_and_reused'] is True
assert decision['readout_layer'] == 23 and decision['readout_k'] == 16
assert len(decision['selected_l23']) == 16
assert result['cross_layer_arm']['selected_l23'] == decision['selected_l23']
assert result['l23_only_control']['selected_l23'] == decision['selected_l23']
assert formal_states() == expected
print(json.dumps({
    'status': decision['status'],
    'L7_only_direct': decision['l7_only_direct_accuracy'],
    'L23_only_direct': decision['l23_only_direct_accuracy'],
    'L7_plus_L23_direct': decision['cross_layer_direct_accuracy'],
    'L7_plus_L23_ranking': decision['cross_layer_ranking_accuracy'],
    'L7_plus_L23_later_token_top1': decision['cross_layer_later_token_top1_accuracy'],
    'L23_only_later_token_top1': decision['l23_only_later_token_top1_accuracy'],
    'direct_synergy': decision['direct_synergy_over_best_control'],
    'selected_l23': decision['selected_l23'],
    'l23_gradient_mass_at_k': decision['l23_gradient_mass_at_k'],
    'l23_effective_count': decision['l23_effective_count'],
    'formal_seed_states': formal_states(),
}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_cross_layer_readout_001.py', '--branch', BRANCH])
assert formal_states() == expected
print(json.dumps({'published': True, 'formal_seed_states': formal_states()}, indent=2))
